# Hyperliquid OFI Scalper Strategy Research
This notebook implements and verifies the Order Flow Imbalance (OFI) Scalper strategy, incorporating a Triple Barrier Method for labeling and a Random Forest meta-labeler to filter trades.

## 1. Import Modules and Setup Paths

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Ensure src directory is in path
sys.path.append(os.path.abspath('../'))

from src.bars import build_volume_bars, map_book_updates_to_bars
from src.features import compute_bar_features
from src.backtester import apply_triple_barriers, simulate_backtest
from src.meta_labeler import purge_and_embargo, train_meta_labeler


## 2. Ingest Data and Build Volume Bars

In [ ]:
trades_file = "../data/raw_trades_BTC.jsonl"
book_file = "../data/raw_book_BTC.jsonl"

# Read trades and check volume
trades_df = pd.read_json(trades_file, lines=True)
total_vol = trades_df["sz"].astype(float).sum()
print(f"Total Trades: {len(trades_df)}")
print(f"Total Traded Volume: {total_vol:.5f} BTC")

# Dynamic volume threshold to get exactly 40 bars on our test sample
v_thresh = total_vol / 40
print(f"Setting Volume Threshold: {v_thresh:.5f} BTC")

bars_df = build_volume_bars(trades_file, v_thresh)
print(f"Created {len(bars_df)} volume bars.")
bars_df.head()

## 3. Map L2 Book Snapshots and Compute Features

In [ ]:
book_bar_mapping = map_book_updates_to_bars(book_file, bars_df)
print(f"Mapped {len(book_bar_mapping)} book updates.")

# Compute level-5 OFI and other bar features
df_features = compute_bar_features(bars_df, book_bar_mapping, levels_count=5, z_window=10)
df_features[["bar_index", "cofi", "cofi_z", "volatility", "volatility_ratio", "depth_ratio"]].head()

## 4. Generate Signal Events (Triple Barrier)

In [ ]:
# Dynamically search for a z-threshold to ensure we get some events for testing
z_threshold = 0.5
events_df = pd.DataFrame()
for thresh in [0.5, 0.2, 0.1, 0.05, 0.01, 0.0]:
    events_df = apply_triple_barriers(df_features, pt_mult=2.0, sl_mult=1.0, hold_bars=5, z_threshold=thresh, execution_mode="maker")
    if len(events_df) >= 3:
        z_threshold = thresh
        break

print(f"Generated {len(events_df)} maker events using z_threshold={z_threshold}.")
if not events_df.empty:
    display(events_df.head())


## 5. Chronological Train-Validation Split and Purge/Embargo

In [ ]:
n_bars = len(df_features)
train_end = int(n_bars * 0.6)
val_start = train_end + 1

print(f"Train range: index 0 to {train_end}. Val range: index >= {val_start}")

if not events_df.empty:
    train_events_purged = purge_and_embargo(
        df_features, events_df, 
        train_start_idx=0, train_end_idx=train_end, 
        val_start_idx=val_start, embargo_pct=0.05
    )
    print(f"Train events count: {len(events_df[events_df['entry_idx'] <= train_end])} raw -> {len(train_events_purged)} purged/embargoed.")
else:
    train_events_purged = pd.DataFrame()
    print("No events to purge.")

## 6. Train Meta-Labeler and Run Backtests

In [ ]:
if len(train_events_purged) > 0:
    meta_model = train_meta_labeler(df_features, train_events_purged)
    print("Model trained.")
else:
    meta_model = None
    print("Using dummy model simulation due to insufficient data.")

if not events_df.empty:
    val_events = events_df[events_df["entry_idx"] >= val_start].copy()
else:
    val_events = pd.DataFrame()

# Run baseline and meta-labeled maker backtests
baseline_res = simulate_backtest(df_features, val_events, maker_fee=0.00015, taker_fee=0.00045, slippage=0.0001)
meta_res = simulate_backtest(df_features, val_events, maker_fee=0.00015, taker_fee=0.00045, slippage=0.0001, meta_model=meta_model, prob_thresh=0.5)

# Benchmark Buy-and-Hold Return
test_bars = df_features[df_features["bar_index"] >= val_start]
if not test_bars.empty and len(test_bars) > 1:
    bench_ret = (test_bars.iloc[-1]["close"] - test_bars.iloc[0]["close"]) / test_bars.iloc[0]["close"]
else:
    bench_ret = 0.0

print("=======================================================")
print("                PERFORMANCE METRICS REPORT             ")
print("=======================================================")
print(f"Benchmark Buy-and-Hold Return: {bench_ret * 100:.4f}%")
print("-------------------------------------------------------")
print("Baseline Maker-Maker/Taker Strategy (Primary Signal Only):")
for k, v in baseline_res["metrics"].items():
    print(f"  - {k}: {v}")
print("-------------------------------------------------------")
print("Meta-Labeled Maker-Maker/Taker Strategy (Filtered):")
for k, v in meta_res["metrics"].items():
    print(f"  - {k}: {v}")
print("=======================================================")


## 7. Data Leakage and Integrity Checks

In [ ]:
leak_detected = False
for idx, row in df_features.iterrows():
    if idx < len(df_features) - 1:
        expected_next_ret = np.log(df_features.loc[idx+1, "close"] / df_features.loc[idx, "close"])
        if not np.isclose(row["next_ret"], expected_next_ret):
            print(f"LEAKAGE DETECTED at index {idx}!")
            leak_detected = True
            
if not leak_detected:
    print("PASS: No future data leakage detected.")
else:
    print("FAIL: Leakage check failed!")